# 🧠 Notebook 04 — CNN-LSTM Model (Main Model)
> **Purpose:** Define, train, and save the primary CNN-LSTM architecture.

```
Input (B, 20, 42)
     │
  ┌──▼─────────────────────┐
  │  1-D CNN (across time)  │  3 × Conv1d → BatchNorm → ReLU → MaxPool
  └──────────────┬──────────┘
                 │ (B, C, T')
  ┌──────────────▼──────────┐
  │  Bi-LSTM  (2 layers)    │
  └──────────────┬──────────┘
                 │ last hidden state
  ┌──────────────▼──────────┐
  │  FC → Dropout → Softmax  │  → 3 classes
  └─────────────────────────┘
```

**Inputs:** `data/X_*_seq.npy`, `data/y_*_seq.npy`  
**Outputs:** `data/cnn_lstm_model.pt`, `data/cnn_lstm_history.csv`

---

## 4.1  Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
sns.set_theme(style='darkgrid')
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

X_train = np.load('data/X_train_seq.npy')
y_train = np.load('data/y_train_seq.npy')
X_val   = np.load('data/X_val_seq.npy')
y_val   = np.load('data/y_val_seq.npy')
X_test  = np.load('data/X_test_seq.npy')
y_test  = np.load('data/y_test_seq.npy')

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 4.2  CNN-LSTM architecture

In [ ]:
class CNNLSTM(nn.Module):
    """
    CNN-LSTM for LOBSTER LOB sequence classification.

    Architecture:
      - Input   : (B, T=20, F=42)
      - CNN     : 3 convolutional blocks over time dimension
      - BiLSTM  : 2-layer bidirectional LSTM
      - Head    : FC → Dropout → 3-class output
    """
    def __init__(self,
                 input_size: int = 42,
                 seq_len:    int = 20,
                 cnn_channels: list = [64, 128, 256],
                 lstm_hidden: int  = 128,
                 lstm_layers: int  = 2,
                 n_classes:   int  = 3,
                 dropout:     float = 0.3):
        super().__init__()

        # ── CNN blocks ──
        cnn_layers = []
        in_ch = input_size
        for out_ch in cnn_channels:
            cnn_layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
                nn.Dropout(dropout / 2)
            ]
            in_ch = out_ch
        self.cnn = nn.Sequential(*cnn_layers)

        # ── Bi-LSTM ──
        self.lstm = nn.LSTM(
            input_size  = cnn_channels[-1],
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            batch_first = True,
            bidirectional = True,
            dropout = dropout if lstm_layers > 1 else 0
        )

        # ── Classification head ──
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),  # ×2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x : (B, T, F)
        x = x.permute(0, 2, 1)          # → (B, F, T)  for Conv1d
        x = self.cnn(x)                  # → (B, C, T)
        x = x.permute(0, 2, 1)          # → (B, T, C)  for LSTM
        out, _ = self.lstm(x)            # → (B, T, 2H)
        out = out[:, -1, :]              # last timestep → (B, 2H)
        return self.head(out)            # → (B, n_classes)


model = CNNLSTM().to(DEVICE)

# Print parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CNN-LSTM  Total trainable parameters: {total_params:,}')
print(model)

## 4.3  Class-weighted loss & optimiser

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes  = np.array([0, 1, 2])
weights  = compute_class_weight('balanced', classes=classes, y=y_train)
w_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print(f'Class weights: DOWN={weights[0]:.3f}  FLAT={weights[1]:.3f}  UP={weights[2]:.3f}')

criterion = nn.CrossEntropyLoss(weight=w_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

## 4.4  DataLoaders

In [ ]:
def make_loader(X, y, batch_size=256, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt),
                      batch_size=batch_size, shuffle=shuffle,
                      pin_memory=(DEVICE.type=='cuda'), num_workers=0)

train_loader = make_loader(X_train, y_train)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)
print(f'Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

## 4.5  Training loop (with early stopping)

In [ ]:
EPOCHS        = 40
PATIENCE      = 7
best_val_f1   = 0.0
patience_cnt  = 0
history       = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):

    # ── Train ──
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # ── Validate ──
    model.eval()
    val_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(Xb)
            val_loss += criterion(logits, yb).item()
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(yb.cpu().numpy())

    val_acc = accuracy_score(labels_all, preds_all)
    val_f1  = f1_score(labels_all, preds_all, average='macro')

    history['train_loss'].append(total_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Ep {epoch:3d}/{EPOCHS} | '
          f'TrLoss {total_loss/len(train_loader):.4f} | '
          f'VaLoss {val_loss/len(val_loader):.4f} | '
          f'Acc {val_acc:.4f} | F1 {val_f1:.4f}')

    # ── Early stopping ──
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'data/cnn_lstm_best.pt')
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (best F1={best_val_f1:.4f})')
            break

print(f'\nBest validation F1-macro: {best_val_f1:.4f}')

## 4.6  Training curves

In [ ]:
hist_df = pd.DataFrame(history)
hist_df.to_csv('data/cnn_lstm_history.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(hist_df['train_loss'], label='Train')
axes[0].plot(hist_df['val_loss'],   label='Val')
axes[0].set_title('CNN-LSTM Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(hist_df['val_acc'], color='steelblue')
axes[1].axhline(0.60, ls='--', color='red', alpha=0.6, label='Target 60%')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(hist_df['val_f1'], color='seagreen')
axes[2].axhline(0.59, ls='--', color='red', alpha=0.6, label='Target F1=0.59')
axes[2].set_title('Validation F1-macro'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.suptitle('CNN-LSTM Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_cnn_lstm_training.png', dpi=120, bbox_inches='tight')
plt.show()

## 4.7  Test set evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('data/cnn_lstm_best.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(DEVICE)
        logits = model(Xb)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())
        all_probs.extend(probs)

y_pred   = np.array(all_preds)
y_probs  = np.array(all_probs)
acc_cnn  = accuracy_score(y_test, y_pred)
f1_cnn   = f1_score(y_test, y_pred, average='macro')

print(f'CNN-LSTM  Test Accuracy: {acc_cnn:.4f}  F1-macro: {f1_cnn:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['DOWN','FLAT','UP']))

# Save predictions
np.save('data/cnn_lstm_preds.npy',  y_pred)
np.save('data/cnn_lstm_probs.npy',  y_probs)
torch.save(model.state_dict(), 'data/cnn_lstm_model.pt')
print('Predictions and model saved.')

## 4.8  Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['DOWN','FLAT','UP'],
            yticklabels=['DOWN','FLAT','UP'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('CNN-LSTM — Confusion Matrix (row-normalised)')
plt.tight_layout()
plt.savefig('data/fig_confusion_cnn_lstm.png', dpi=120, bbox_inches='tight')
plt.show()

---
> ✅ **CNN-LSTM trained.** Proceed to `05_evaluation_and_comparison.ipynb`.